In [8]:
import pandas as pd
import jieba
import re
from collections import Counter
import os
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
def get_word_freq(df, filename, n=300, stopwords_path='哈工大停用词表.txt', sensitive_names_path='llm_sensitive_name.csv', text_col='text'):
    # 检查文件是否存在
    if not os.path.exists(stopwords_path):
        raise FileNotFoundError(f"停用词文件不存在: {stopwords_path}")
    if not os.path.exists(sensitive_names_path):
        raise FileNotFoundError(f"敏感词文件不存在: {sensitive_names_path}")
    
    # 读取停用词
    with open(stopwords_path, 'r', encoding='utf-8') as f:
        stopwords = set(line.strip() for line in f if line.strip())
    # 读取敏感词/人名表
    sen_names = set(pd.read_csv(sensitive_names_path)['name'].dropna().astype(str).str.strip().tolist())
    for name in sen_names:
        jieba.add_word(name)
    
    def clean_text(text):
        text = str(text)
        text = re.sub(r'[A-Za-z0-9]+', ' ', text)
        text = re.sub(r'[^\u4e00-\u9fa5]', ' ', text)
        return text
    
    texts = df[text_col].dropna().astype(str).tolist()
    
    all_words = []
    for text in texts:
        text = clean_text(text)
        words = jieba.lcut(text)
        words = [
            w for w in words
            if len(w.strip()) > 1
            and w not in stopwords
            and w not in sen_names
        ]
        all_words.extend(words)
    
    word_freq = Counter(all_words)
    temp = pd.DataFrame(word_freq.most_common(n), columns=['word', 'freq'])
    temp.to_excel('./analysis-data/word_cloud/' + filename + '.xlsx', index=None)
    return temp

In [10]:
def get_tfidf(
    df,
    filename,
    n=300,
    stopwords_path='哈工大停用词表.txt',
    sensitive_names_path='llm_sensitive_name.csv',
    text_col='text',
    output_dir='./analysis-data/word_cloud/',
    min_df=2,
    max_df=0.8,
    agg='sum'
):
    if text_col not in df.columns:
        raise ValueError(f"DataFrame 中不存在列: {text_col}")

    if not os.path.exists(stopwords_path):
        raise FileNotFoundError(f"停用词文件不存在: {stopwords_path}")

    if not os.path.exists(sensitive_names_path):
        raise FileNotFoundError(f"敏感词文件不存在: {sensitive_names_path}")

    if agg not in ['sum', 'mean']:
        raise ValueError("agg 只能是 'sum' 或 'mean'")

    # 读取停用词
    with open(stopwords_path, 'r', encoding='utf-8') as f:
        stopwords = set(line.strip() for line in f if line.strip())

    # 读取敏感词
    sensitive_df = pd.read_csv(sensitive_names_path)
    if 'name' not in sensitive_df.columns:
        raise ValueError("敏感词文件中必须包含 'name' 列")

    sen_names = set(
        sensitive_df['name']
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )

    for name in sen_names:
        jieba.add_word(name)

    def clean_text(text):
        text = str(text)
        text = re.sub(r'[A-Za-z0-9]+', ' ', text)      # 去英文和数字
        text = re.sub(r'[^\u4e00-\u9fa5]', ' ', text)  # 仅保留中文
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def tokenize(text):
        text = clean_text(text)
        words = jieba.lcut(text)
        words = [
            w.strip() for w in words
            if len(w.strip()) > 1
            and w.strip() not in stopwords
            and w.strip() not in sen_names
        ]
        return words

    texts = df[text_col].dropna().astype(str).tolist()

    # corpus: 给 TfidfVectorizer 用
    # tokenized_docs: 给 freq / doc_count 用
    corpus = []
    tokenized_docs = []

    for text in texts:
        words = tokenize(text)
        if words:
            tokenized_docs.append(words)
            corpus.append(' '.join(words))




    total_counter = Counter()
    for words in tokenized_docs:
        total_counter.update(words)

 
    doc_counter = Counter()
    for words in tokenized_docs:
        doc_counter.update(set(words))


    vectorizer = TfidfVectorizer(
        tokenizer=str.split,
        token_pattern=None,
        min_df=min_df,
        max_df=max_df
    )

    tfidf_matrix = vectorizer.fit_transform(corpus)
    feature_names = vectorizer.get_feature_names_out()

    if len(feature_names) == 0:
        raise ValueError("没有词通过 min_df/max_df 过滤条件，请调整参数。")

    if agg == 'sum':
        scores = tfidf_matrix.sum(axis=0).A1
    else:  # agg == 'mean'
        scores = tfidf_matrix.mean(axis=0).A1


    result = pd.DataFrame({
        'word': feature_names,
        'tfidf': scores
    })

    result['freq'] = result['word'].map(total_counter).fillna(0).astype(int)
    result['doc_count'] = result['word'].map(doc_counter).fillna(0).astype(int)

 
    result = result[['word', 'freq', 'doc_count', 'tfidf']]

   
    result = result.sort_values(by='tfidf', ascending=False).head(n).reset_index(drop=True)

    # 保存
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f'{filename}')
    result.to_excel(output_path + '.xlsx', index=False)
    result.to_csv(output_path + '.csv', index=False)

    return result

In [11]:
local = pd.read_csv('./data/topic-results/result_local_topic.csv')

In [12]:
get_tfidf(local, 'local_TF-IDF')

,word,freq,doc_count,tfidf
0,发展,4390,941,68.054155
1,旅游,3102,586,58.030188
2,中心,1936,698,48.921188
3,检测,1801,334,48.400361
4,感染,1215,401,47.502118
...,...,...,...,...
295,打卡,295,154,9.916789
296,因应,372,310,9.895629
297,指出,326,223,9.871030
298,学校,285,121,9.848308


In [13]:
non = pd.read_csv('./data/topic-results/result_non_local.csv')
get_tfidf(non, 'non_TF-IDF')

,word,freq,doc_count,tfidf
0,发展,90869,2994,342.778789
1,社会主义,48982,1516,248.003498
2,全面,50119,2429,225.839130
3,坚持,54779,1896,224.989261
4,建设,57034,2666,210.669143
...,...,...,...,...
295,行动,4646,1165,25.884799
296,单位,3061,1413,25.855667
297,现场,2625,1020,25.829251
298,依法,5416,1340,25.824824


In [16]:
get_tfidf(local[local.label.str.contains('文化与旅游')], 'local_culture_TF-IDF')

,word,freq,doc_count,tfidf
0,旅游,2847,443,39.783344
1,旅游局,1439,317,26.285457
2,活动,1911,473,26.108668
3,发展,1628,358,24.863831
4,旅客,1242,342,22.159430
...,...,...,...,...
295,团队,116,60,3.771482
296,摊位,113,64,3.750872
297,游客,103,65,3.749116
298,呈献,123,88,3.742116


In [17]:
get_tfidf(non[non.label.str.contains('文化与旅游')], 'non_culture_TF-IDF')

,word,freq,doc_count,tfidf
0,发展,12111,655,41.843422
1,珠海,2772,313,39.429727
2,中国,8532,772,31.344914
3,文化,6022,680,29.265650
4,旅游,3046,650,25.751449
...,...,...,...,...
295,朋友,498,236,5.476015
296,出游,236,100,5.471317
297,两地,297,138,5.462955
298,业务,283,147,5.447208
